# FNI-LLM Training Notebook
## Cameroon Language Model — GPU Training

This notebook trains the FNI-LLM Transformer on Cameroon language data using Colab GPU.

**Languages:** English → French → Bayangi → Douala


In [ ]:
# Cell 1: Mount Drive and sync code
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/FNI_AI_LLM
!git pull origin master
print('Code synced!')

In [ ]:
# Cell 2: Install dependencies and check GPU
!pip install torch torchvision --quiet

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
!nvidia-smi

In [ ]:
# Cell 3: Setup path
import sys
sys.path.insert(0, '/content/drive/MyDrive/FNI_AI_LLM')
print('Path set up!')

In [ ]:
# Cell 4: PyTorch Transformer Model
import torch
import torch.nn as nn
import numpy as np

class FNITransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_heads=4,
                 d_ff=512, num_layers=4, max_seq_len=64, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=d_ff, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output = nn.Linear(d_model, vocab_size)
        self.d_model = d_model
        self.vocab_size = vocab_size

    def forward(self, x):
        seq_len = x.shape[1]
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(positions)
        x = self.transformer(x)
        return self.output(x)

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

print('FNITransformer defined!')

In [ ]:
# Cell 5: Load data
import sys
sys.path.insert(0, '/content/drive/MyDrive/FNI_AI_LLM')

from src.year3.data_processing.dataset import TextDataset
from src.year3.data_processing.dataloaders import DataLoader, train_val_test_split

LANGUAGE  = 'english'   # Change to 'french', 'bayangi', 'douala'
SEQ_LEN   = 32
DATA_PATH = f'data/cameroon_languages/{LANGUAGE}/processed/{LANGUAGE}_clean.txt'

ds = TextDataset(language=LANGUAGE, seq_len=SEQ_LEN)
ds.load(DATA_PATH)
train_ds, val_ds, test_ds = train_val_test_split(ds, seed=42)

print(f'Dataset: {ds}')
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# Cell 6: Training configuration
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
VOCAB_SIZE = ds.vocab_size
D_MODEL    = 128
NUM_HEADS  = 4
D_FF       = 512
NUM_LAYERS = 4
BATCH_SIZE = 64
EPOCHS     = 20
LR         = 3e-4

model = FNITransformer(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL,
    num_heads=NUM_HEADS, d_ff=D_FF,
    num_layers=NUM_LAYERS, max_seq_len=SEQ_LEN
).to(DEVICE)

print(f'Model parameters: {model.count_params():,}')
print(f'Device: {DEVICE}')
print(f'Vocab size: {VOCAB_SIZE}')

In [ ]:
# Cell 7: Training loop
import time
import json
import os

optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion  = nn.CrossEntropyLoss(ignore_index=0)  # ignore PAD token

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

best_val_loss = float('inf')
log = []
os.makedirs('models/checkpoints', exist_ok=True)
os.makedirs('logs', exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    # --- TRAIN ---
    model.train()
    train_losses = []
    t0 = time.time()

    for X_batch, y_batch in train_loader:
        X = torch.tensor(X_batch, dtype=torch.long).to(DEVICE)
        y = torch.tensor(y_batch, dtype=torch.long).to(DEVICE)

        optimizer.zero_grad()
        logits = model(X)                          # (batch, seq, vocab)
        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),        # (batch*seq, vocab)
            y.reshape(-1)                          # (batch*seq,)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_losses.append(loss.item())

    # --- VALIDATE ---
    model.eval()
    val_losses = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X = torch.tensor(X_batch, dtype=torch.long).to(DEVICE)
            y = torch.tensor(y_batch, dtype=torch.long).to(DEVICE)
            logits = model(X)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))
            val_losses.append(loss.item())

    train_loss = np.mean(train_losses)
    val_loss   = np.mean(val_losses)
    elapsed    = time.time() - t0
    scheduler.step()

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(),
                   f'models/checkpoints/{LANGUAGE}_best.pt')

    entry = dict(epoch=epoch, train_loss=round(float(train_loss), 4),
                 val_loss=round(float(val_loss), 4), time=round(elapsed, 2))
    log.append(entry)
    print(f'Epoch {epoch:3d}/{EPOCHS} | '
          f'train={train_loss:.4f} | val={val_loss:.4f} | '
          f'time={elapsed:.1f}s')

with open(f'logs/{LANGUAGE}_colab_training.json', 'w') as f:
    json.dump(log, f, indent=2)

print(f'\nBest val loss: {best_val_loss:.4f}')
print(f'Model saved: models/checkpoints/{LANGUAGE}_best.pt')

In [ ]:
# Cell 8: Plot training curves
import matplotlib.pyplot as plt

train_losses = [e['train_loss'] for e in log]
val_losses   = [e['val_loss']   for e in log]

plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.title(f'FNI-LLM Training — {LANGUAGE.capitalize()}')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(f'docs/visualizations/{LANGUAGE}_training_curve.png', dpi=100)
plt.show()
print('Training curve saved!')

In [ ]:
# Cell 9: Generate text from trained model
def generate(model, tokenizer, prompt, max_new_tokens=20,
             temperature=0.8, device='cpu'):
    model.eval()
    ids = tokenizer.encode(prompt)
    seen = {}

    with torch.no_grad():
        for _ in range(max_new_tokens):
            x = torch.tensor([ids], dtype=torch.long).to(device)
            logits = model(x)[0, -1].cpu().numpy()

            # Repetition penalty
            for tid, count in seen.items():
                logits[tid] -= 1.5 * count

            # Temperature sampling
            logits = logits / temperature
            probs  = np.exp(logits - np.max(logits))
            probs  = probs / probs.sum()
            next_id = int(np.random.choice(len(probs), p=probs))
            seen[next_id] = seen.get(next_id, 0) + 1
            ids.append(next_id)

    return tokenizer.decode(ids)

# Load best model
model.load_state_dict(
    torch.load(f'models/checkpoints/{LANGUAGE}_best.pt',
               map_location=DEVICE))

tokenizer = ds.tokenizer
prompts = ['cameroon is', 'the language', 'douala is']

print(f'=== GENERATED TEXT ({LANGUAGE}) ===')
for p in prompts:
    out = generate(model, tokenizer, p, max_new_tokens=15,
                   temperature=0.8, device=DEVICE)
    print(f'Prompt: "{p}"')
    print(f'Output: "{out}"')
    print()

In [ ]:
# Cell 10: Push results back to GitHub
%cd /content/drive/MyDrive/FNI_AI_LLM
!git add logs/ docs/visualizations/
!git commit -m '[Year3] Training complete - {LANGUAGE} model trained on Colab GPU'
!git push origin master
print('Results pushed to GitHub!')